# Valencia Properties — Price Prediction Model

Goal: predict the fair market price for each listing → flag undervalued ("bargain") and overvalued listings.

**Approach**
- Two separate **LightGBM** regressors: one for **Casas** (chalets/townhouses), one for **Pisos** (flats/apartments)
- Target: `log(price_eur)` (right-skewed distribution)
- LightGBM handles missing values and categorical features natively
- Evaluate with MAE, MAPE, median APE, R² on a held-out test set
- Use **SHAP** to explain what drives predictions

**Sections**
1. Setup & data
2. Train Casa model
3. Train Piso model
4. Predicted vs actual & residuals
5. SHAP explanations
6. Bargain leaderboards
7. Bargain map

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import lightgbm as lgb
import shap
import joblib

from models.price_model import (
    prepare_features, train_one_model, score_dataset,
    NUMERIC_FEATURES, BOOLEAN_FEATURES, CATEGORICAL_FEATURES, ALL_FEATURES,
)

shap.initjs()
pd.set_option("display.max_columns", 60)

df = pd.read_parquet("../valencia_clean.parquet")
print(f"Loaded {len(df):,} rows")
print(f"Casas: {(df['property_type']=='Casa').sum()} | Pisos: {(df['property_type']=='Piso').sum()}")

## 1. Train models

Reuses the same training function used by `models/price_model.py` so results are reproducible.

In [ ]:
casa_df = df[df['property_type'] == 'Casa']
piso_df = df[df['property_type'] == 'Piso']

casa_model, casa_metrics, casa_imp = train_one_model(casa_df, 'Casa')
piso_model, piso_metrics, piso_imp = train_one_model(piso_df, 'Piso')

metrics_df = pd.DataFrame([casa_metrics, piso_metrics]).set_index('name')
metrics_df

## 2. Predicted vs actual & residuals

How tightly do predictions track real prices?

In [ ]:
def diagnostic_plots(df_subset, model, name):
    X, y = prepare_features(df_subset)
    pred_log = model.predict(X)
    pred = np.exp(pred_log)
    actual = np.exp(y).values

    residuals_pct = (pred - actual) / actual * 100

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(f"{name}: predicted vs actual (log-log)",
                        f"{name}: residuals (% error)"),
    )

    fig.add_trace(
        go.Scatter(x=actual, y=pred, mode="markers",
                   marker=dict(size=4, opacity=0.4, color="#3b82f6"),
                   name=name),
        row=1, col=1,
    )
    diag = np.linspace(actual.min(), actual.max(), 100)
    fig.add_trace(
        go.Scatter(x=diag, y=diag, mode="lines",
                   line=dict(color="red", dash="dash"), name="y=x"),
        row=1, col=1,
    )

    fig.add_trace(
        go.Histogram(x=residuals_pct.clip(-100, 100), nbinsx=50,
                     marker_color="#f59e0b", name="error %"),
        row=1, col=2,
    )

    fig.update_xaxes(type="log", row=1, col=1)
    fig.update_yaxes(type="log", row=1, col=1)
    fig.update_layout(height=420, showlegend=False, title_text=name)
    return fig

diagnostic_plots(casa_df, casa_model, "Casa").show()
diagnostic_plots(piso_df, piso_model, "Piso").show()

## 3. Feature importance (LightGBM gain)

Which features contribute most to the splits?

In [ ]:
imp_combined = (
    casa_imp.assign(model="Casa")
    .merge(piso_imp.assign(model="Piso"), on="feature", suffixes=("_casa", "_piso"))
)
imp_long = pd.concat([
    casa_imp.assign(model="Casa"),
    piso_imp.assign(model="Piso"),
])

fig = px.bar(
    imp_long.sort_values("importance", ascending=True),
    x="importance",
    y="feature",
    color="model",
    barmode="group",
    title="Feature importance (LightGBM gain)",
    color_discrete_map={"Casa": "#10b981", "Piso": "#3b82f6"},
)
fig.update_layout(height=600, yaxis={"categoryorder": "total ascending"})
fig.show()

## 4. SHAP explanations

SHAP values quantify each feature's contribution to each individual prediction.
Beeswarm plot shows feature impact direction & magnitude across the test set.

In [ ]:
def shap_summary(df_subset, model, name, sample=1500):
    X, _ = prepare_features(df_subset)
    if len(X) > sample:
        X = X.sample(sample, random_state=42)
    explainer = shap.TreeExplainer(model)
    sv = explainer.shap_values(X)
    print(f"{name} — SHAP summary (effect on log(price))")
    shap.summary_plot(sv, X, max_display=15)

shap_summary(casa_df, casa_model, "Casa")

In [ ]:
shap_summary(piso_df, piso_model, "Piso")

## 5. Bargain leaderboards

Listings priced significantly below their predicted fair value (≥ 15% below).

In [ ]:
scored = score_dataset(df, casa_model, piso_model)

print(f"Total bargains (≥15% below predicted): {scored['is_bargain'].sum()}")
print(f"Total overpriced (≥15% above predicted): {scored['is_overpriced'].sum()}")

display_cols = [
    "property_subtype", "city", "district",
    "price_eur", "predicted_price", "bargain_pct",
    "surface_m2", "bedrooms", "feature_count",
    "listing_age_days", "url",
]

print("\nTop 15 PISO bargains (≥50m², price ≥ €40k, listed ≤180 days):")
piso_b = scored[(scored["property_type"] == "Piso") &
                (scored["surface_m2"] >= 50) &
                (scored["price_eur"] >= 40000) &
                (scored["listing_age_days"] <= 180) &
                (scored["is_bargain"])]
display(piso_b.nlargest(15, "bargain_pct")[display_cols].reset_index(drop=True))

In [ ]:
print("Top 15 CASA bargains (≥80m², price ≥ €50k, listed ≤180 days):")
casa_b = scored[(scored["property_type"] == "Casa") &
                (scored["surface_m2"] >= 80) &
                (scored["price_eur"] >= 50000) &
                (scored["listing_age_days"] <= 180) &
                (scored["is_bargain"])]
display(casa_b.nlargest(15, "bargain_pct")[display_cols].reset_index(drop=True))

## 6. Bargain map

Geographic view of bargains. Color = bargain %, size = surface.

In [ ]:
bargain_map = scored[scored["is_bargain"] & (scored["surface_m2"] >= 50)]

fig = px.scatter_mapbox(
    bargain_map,
    lat="latitude",
    lon="longitude",
    color="bargain_pct",
    size="surface_m2",
    size_max=14,
    hover_data={
        "city": True,
        "property_subtype": True,
        "price_eur": ":,.0f",
        "predicted_price": ":,.0f",
        "bargain_pct": ":.1f",
        "surface_m2": True,
        "bedrooms": True,
    },
    color_continuous_scale="Greens",
    zoom=8,
    center={"lat": 39.4699, "lon": -0.3763},
    title=f"Bargain map — {len(bargain_map)} listings priced ≥15% below predicted",
    height=650,
)
fig.update_layout(mapbox_style="open-street-map", margin={"r": 0, "t": 40, "l": 0, "b": 0})
fig.show()

## 7. Caveats & next steps

**Caveats**
- Bargains may reflect **unobserved features**: bad condition, structural issues, legal problems, no light, top-floor walk-up, etc. The model can only see what's in the data.
- The **listing description** (free text) is not used. NLP on description could improve precision.
- Casa MAPE is high (27%) — chalets are more idiosyncratic. Consider sub-segmenting by surface bins or location.
- Some "bargains" are likely **stale price errors** or listings the seller forgot to update — cross-reference with `listing_age_days`.

**Next**
- Use the trained models in the Streamlit dashboard (Phase 4) for live filtering by bargain status.
- Add price tracking (Phase 5) to see if "bargains" actually sell faster.